# MVP08 — Análise de Dados

Aqui seram respondidas às sete perguntas definidas no **MVP00-objetivo**, consultando exclusivamente a camada Gold.

Cada pergunta segue a mesma estrutura:
1. **Pergunta** e como ela será medida.
2. **Consulta** sobre o modelo dimensional.
3. **Visualização**.
4. **Resultado e interpretação**, preenchidos após a execução, sempre em termos do problema de negócio.

---
## Pergunta 1
**Quanto varia a nota média da avaliação quando o pedido é entregue após a data estimada, quando o efeito se torna severo?**

Medida: nota média e percentual de notas baixas (1 ou 2) por faixa de atraso.

In [0]:
%sql
SELECT
  CASE
    WHEN dias_atraso <= -10 THEN '1. Antecipado 10+ dias'
    WHEN dias_atraso <= 0   THEN '2. No prazo (até 9 dias antes)'
    WHEN dias_atraso <= 3   THEN '3. Atraso 1-3 dias'
    WHEN dias_atraso <= 7   THEN '4. Atraso 4-7 dias'
    WHEN dias_atraso <= 15  THEN '5. Atraso 8-15 dias'
    ELSE                         '6. Atraso 16+ dias'
  END                                                                     AS faixa_atraso,
  COUNT(*)                                                                AS pedidos,
  ROUND(AVG(nota_avaliacao), 2)                                           AS nota_media,
  ROUND(AVG(CASE WHEN nota_avaliacao <= 2 THEN 1.0 ELSE 0 END) * 100, 1)  AS pct_notas_baixas,
  ROUND(AVG(CASE WHEN nota_avaliacao = 5  THEN 1.0 ELSE 0 END) * 100, 1)  AS pct_nota_5
FROM mvp.gold.fato_pedidos
WHERE dias_atraso IS NOT NULL AND nota_avaliacao IS NOT NULL
GROUP BY 1
ORDER BY 1;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Visão resumida: no prazo × atrasado
SELECT
  CASE WHEN flag_atraso THEN 'Atrasado' ELSE 'No prazo' END              AS situacao,
  COUNT(*)                                                               AS pedidos,
  ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1)                       AS pct_pedidos,
  ROUND(AVG(nota_avaliacao), 2)                                          AS nota_media,
  ROUND(AVG(CASE WHEN nota_avaliacao <= 2 THEN 1.0 ELSE 0 END) * 100, 1) AS pct_notas_baixas
FROM mvp.gold.fato_pedidos
WHERE flag_atraso IS NOT NULL AND nota_avaliacao IS NOT NULL
GROUP BY 1;

**Resultado e interpretação — P1**

Pedidos entregues no prazo têm nota média de **4.29**, contra **2.27** nos atrasados (queda de **2.02** pontos).
O percentual de notas baixas passa de **9.3 %** para **62.4 %**. O efeito se torna severo a partir da faixa **"4. Atraso 4-7 dias"**,
onde **As notas baixas representam mais de 50% dos casos, e a nota média é inferior a 3. Além disso, a quantidade de pedidos é semelhante entre as faixas mais próximas**.

---
## Pergunta 2
**Quais estados concentram os maiores prazos de entrega e qual o peso do frete sobre o valor do pedido em cada um?**

Medida: prazo médio e mediano, percentual médio do frete sobre o valor total e percentual de pedidos atrasados, por UF do cliente.

In [0]:
%sql
SELECT
  c.uf,
  c.regiao,
  COUNT(*)                                                                         AS pedidos,
  ROUND(AVG(f.dias_entrega), 1)                                                    AS prazo_medio_dias,
  percentile_approx(f.dias_entrega, 0.5)                                           AS prazo_mediano_dias,
  ROUND(AVG(CAST(f.valor_frete AS DOUBLE) / NULLIF(CAST(f.valor_total AS DOUBLE), 0)) * 100, 1) AS pct_frete_medio,
  ROUND(AVG(CASE WHEN f.flag_atraso THEN 1.0 ELSE 0 END) * 100, 1)                 AS pct_atrasados
FROM mvp.gold.fato_pedidos f
JOIN mvp.gold.dim_cliente c ON f.customer_id = c.customer_id
WHERE f.dias_entrega IS NOT NULL AND f.valor_total IS NOT NULL
GROUP BY c.uf, c.regiao
ORDER BY prazo_medio_dias DESC;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Visão por região
SELECT
  c.regiao,
  COUNT(*)                                                                         AS pedidos,
  ROUND(AVG(f.dias_entrega), 1)                                                    AS prazo_medio_dias,
  ROUND(AVG(CAST(f.valor_frete AS DOUBLE) / NULLIF(CAST(f.valor_total AS DOUBLE), 0)) * 100, 1) AS pct_frete_medio
FROM mvp.gold.fato_pedidos f
JOIN mvp.gold.dim_cliente c ON f.customer_id = c.customer_id
WHERE f.dias_entrega IS NOT NULL AND f.valor_total IS NOT NULL
GROUP BY c.regiao
ORDER BY prazo_medio_dias DESC;

**Resultado e interpretação — P2**

- Os maiores prazos estão em **RR**, com média de **29.3 dias**, contra **8.7 dias** em **SP**.
- Nessas UFs o frete representa **28.3%** do valor do pedido, contra **19.5%** no Sudeste.
- Os clientes do Norte esperam mais que os do Sudeste, mas a quantidade do pedidos no Sudeste e bem maior do que Norte.

---
## Pergunta 3
**Pedidos interestaduais (vendedor e cliente em estados diferentes) demoram significativamente mais que os intraestaduais?**

Medida: comparação de prazo, atraso e nota entre os dois grupos, seguida de teste estatístico.

In [0]:
%sql
SELECT
  CASE WHEN flag_interestadual THEN 'Interestadual' ELSE 'Intraestadual' END AS tipo_envio,
  COUNT(*)                                                                   AS pedidos,
  ROUND(AVG(dias_entrega), 1)                                                AS prazo_medio_dias,
  percentile_approx(dias_entrega, 0.5)                                       AS prazo_mediano_dias,
  ROUND(AVG(CASE WHEN flag_atraso THEN 1.0 ELSE 0 END) * 100, 1)             AS pct_atrasados,
  ROUND(AVG(nota_avaliacao), 2)                                              AS nota_media
FROM mvp.gold.fato_pedidos
WHERE dias_entrega IS NOT NULL AND flag_interestadual IS NOT NULL
GROUP BY 1;

Databricks visualization. Run in Databricks to view.

**Teste de significância.** Prazos de entrega têm distribuição assimétrica, por isso usa-se o teste não paramétrico de **Mann-Whitney U**, que compara as distribuições sem supor normalidade. Hipótese alternativa unilateral: prazos interestaduais são maiores.

Com dezenas de milhares de pedidos, quase qualquer diferença resulta estatisticamente significativa; por isso a diferença de medianas também é reportada, e é ela que indica relevância prática.

In [0]:
from scipy import stats

df_p3 = spark.sql("""
    SELECT flag_interestadual, dias_entrega
    FROM mvp.gold.fato_pedidos
    WHERE dias_entrega IS NOT NULL AND flag_interestadual IS NOT NULL
""").toPandas()

grupo = df_p3["flag_interestadual"].astype(bool)
inter = df_p3.loc[grupo, "dias_entrega"]
intra = df_p3.loc[~grupo, "dias_entrega"]

u, p_valor = stats.mannwhitneyu(inter, intra, alternative="greater")

print(f"Interestadual: n = {len(inter):,} | mediana = {inter.median():.0f} dias | média = {inter.mean():.1f} dias")
print(f"Intraestadual: n = {len(intra):,} | mediana = {intra.median():.0f} dias | média = {intra.mean():.1f} dias")
print(f"Diferença de medianas: {inter.median() - intra.median():.0f} dias")
print(f"Mann-Whitney U (unilateral): p-valor = {p_valor:.2e}")

**Resultado e interpretação — P3**

Pedidos interestaduais levam em mediana **13 dias**, contra **7 dias** nos intraestaduais (diferença de **6 dias**). O teste de Mann-Whitney resultou em p-valor **<0.001**, portanto a diferença **é** estatisticamente significativa. Como amostras grandes tornam significativas até diferenças mínimas, a relevância foi avaliada também pelo tamanho do efeito: seis dias a mais na mediana é uma diferença de grande impacto prático.

---
## Pergunta 4
**Quais categorias concentram a receita e quais combinam alto volume com avaliações baixas?**

Medida: receita e participação acumulada por categoria (a partir de `fato_itens_pedido`) e nota média por categoria.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_categorias AS
WITH receita AS (
  SELECT p.categoria,
         SUM(i.preco)               AS receita,
         COUNT(DISTINCT i.order_id) AS pedidos
  FROM mvp.gold.fato_itens_pedido i
  JOIN mvp.gold.dim_produto p ON i.product_id = p.product_id
  GROUP BY p.categoria
),
nota AS (
  SELECT categoria,
         AVG(nota_avaliacao)                                           AS nota_media,
         AVG(CASE WHEN nota_avaliacao <= 2 THEN 1.0 ELSE 0 END) * 100  AS pct_notas_baixas
  FROM (
    SELECT DISTINCT i.order_id, p.categoria, f.nota_avaliacao
    FROM mvp.gold.fato_itens_pedido i
    JOIN mvp.gold.dim_produto p  ON i.product_id = p.product_id
    JOIN mvp.gold.fato_pedidos f ON i.order_id   = f.order_id
    WHERE f.nota_avaliacao IS NOT NULL
  ) pedido_categoria
  GROUP BY categoria
)
SELECT r.categoria, r.receita, r.pedidos, n.nota_media, n.pct_notas_baixas
FROM receita r
LEFT JOIN nota n ON r.categoria = n.categoria;

SELECT
  categoria,
  ROUND(receita, 2)                                                            AS receita,
  ROUND(receita / SUM(receita) OVER () * 100, 2)                               AS pct_receita,
  ROUND(SUM(receita) OVER (ORDER BY receita DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(receita) OVER () * 100, 1) AS pct_receita_acumulada,
  pedidos,
  ROUND(nota_media, 2)                                                         AS nota_media,
  ROUND(pct_notas_baixas, 1)                                                   AS pct_notas_baixas
FROM v_categorias
ORDER BY receita DESC;

Databricks visualization. Run in Databricks to view.

**Categorias críticas:** alto volume (quartil superior de pedidos) **e** nota média abaixo da média geral da plataforma.

In [0]:
%sql
SELECT
  categoria,
  pedidos,
  ROUND(nota_media, 2)       AS nota_media,
  ROUND(pct_notas_baixas, 1) AS pct_notas_baixas,
  ROUND(receita, 2)          AS receita
FROM v_categorias
WHERE pedidos    >= (SELECT percentile_approx(pedidos, 0.75) FROM v_categorias)
  AND nota_media <  (SELECT AVG(nota_avaliacao) FROM mvp.gold.fato_pedidos)
ORDER BY nota_media;

In [0]:
%sql
SELECT ROUND(AVG(nota_avaliacao), 2) AS nota_media_geral
FROM mvp.gold.fato_pedidos;

**Resultado e interpretação — P4**

As **8** maiores categorias concentram **54.4 %** da receita, lideradas por **beleza_saude** (**9.26 %**).
Entre as categorias de alto volume, **7 categorias (relogios_presentes, bebes, informatica_acessorios, moveis_decoracao, telefonia, cama_mesa_banho e sem_categoria)** ficam abaixo da média geral de nota, com uma media simples entre as categorias do **16.64 %** de notas baixas.

---
## Pergunta 5
**Como evoluíram a receita mensal e o ticket médio entre os anos 2017 e 2018?**

Medida: receita, pedidos e ticket médio por mês, excluindo pedidos cancelados/indisponíveis e os meses de borda com volume residual.

In [0]:
%sql
SELECT
  t.ano_mes,
  COUNT(*)                     AS pedidos,
  ROUND(SUM(f.valor_total), 2) AS receita,
  ROUND(AVG(f.valor_total), 2) AS ticket_medio
FROM mvp.gold.fato_pedidos f
JOIN mvp.gold.dim_tempo t ON f.data_compra = t.data
WHERE f.valor_total IS NOT NULL
  AND f.status_pedido NOT IN ('canceled', 'unavailable')
  AND t.ano_mes BETWEEN '2017-01' AND '2018-08'
GROUP BY t.ano_mes
ORDER BY t.ano_mes;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Comparação ano contra ano no mesmo período (janeiro a agosto)
SELECT
  t.ano,
  COUNT(*)                     AS pedidos,
  ROUND(SUM(f.valor_total), 2) AS receita,
  ROUND(AVG(f.valor_total), 2) AS ticket_medio
FROM mvp.gold.fato_pedidos f
JOIN mvp.gold.dim_tempo t ON f.data_compra = t.data
WHERE f.valor_total IS NOT NULL
  AND f.status_pedido NOT IN ('canceled', 'unavailable')
  AND t.ano IN (2017, 2018)
  AND t.mes BETWEEN 1 AND 8
GROUP BY t.ano
ORDER BY t.ano;

**Resultado e interpretação — P5**

A receita mensal passou de **R$ 136943.46** em jan/2017 para **R$ 996973.51** em ago/2018. Comparando janeiro a agosto, 2018 cresceu **140.36 %** sobre 2017 em receita, enquanto o ticket médio variou **1.31 %**. Isso indica que o crescimento veio **do volume de pedidos**.
No Novembro/2017, as receitas foram as mais altas durante o período analisado.

---
## Pergunta 6
**O pagamento parcelado está relacionado ao tickets mais altos? Qual o meio de pagamento mais frecuente?**

Medida: participação de cada meio de pagamento; ticket médio e mediano por faixa de parcelas; correlação entre parcelas e valor. A análise de parcelas considera apenas **cartão de crédito**, único meio que permite parcelamento (boleto, voucher e débito são sempre à vista e distorceriam a comparação).

In [0]:
%sql
SELECT
  tipo_pagamento_principal,
  COUNT(*)                                         AS pedidos,
  ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct_pedidos,
  ROUND(AVG(valor_total), 2)                       AS ticket_medio
FROM mvp.gold.fato_pedidos
WHERE tipo_pagamento_principal IS NOT NULL AND valor_total IS NOT NULL
GROUP BY tipo_pagamento_principal
ORDER BY pedidos DESC;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
  CASE
    WHEN qtd_parcelas = 1   THEN '01x'
    WHEN qtd_parcelas <= 3  THEN '02-03x'
    WHEN qtd_parcelas <= 6  THEN '04-06x'
    WHEN qtd_parcelas <= 10 THEN '07-10x'
    ELSE                         '11x ou mais'
  END                                                                   AS faixa_parcelas,
  COUNT(*)                                                              AS pedidos,
  ROUND(AVG(valor_total), 2)                                            AS ticket_medio,
  ROUND(percentile_approx(CAST(valor_total AS DOUBLE), 0.5), 2)         AS ticket_mediano
FROM mvp.gold.fato_pedidos
WHERE tipo_pagamento_principal = 'credit_card'
  AND qtd_parcelas IS NOT NULL AND valor_total IS NOT NULL
GROUP BY 1
ORDER BY 1;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT ROUND(corr(qtd_parcelas, CAST(valor_total AS DOUBLE)), 3) AS correlacao_parcelas_valor
FROM mvp.gold.fato_pedidos
WHERE tipo_pagamento_principal = 'credit_card'
  AND qtd_parcelas IS NOT NULL AND valor_total IS NOT NULL;

**Resultado e interpretação — P6**

O meio dominante é **credit_card**, com **75.5 %** dos pedidos. No cartão de crédito, o ticket mediano sobe de **R$ 71.63** (à vista) para **R$ 213.74** (**faixa: 11x ou mais**). A correlação de Pearson entre parcelas e valor é **r = 0.369**, indicando associação **moderada**.
Ressalva: correlação não implica causalidade; é provável que compras caras levem ao parcelamento, e não o contrário.

---
## Pergunta 7
**A receita está distribuída de forma equilibrada entre os vendedores ou concentrada em poucos participantes com elevada representatividade no faturamento?**

Medida: quantos vendedores (e que percentual do total) são necessários para atingir 50% e 80% da receita.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_ranking_vendedores AS
SELECT
  seller_id,
  receita,
  ROW_NUMBER() OVER (ORDER BY receita DESC)                                     AS posicao,
  COUNT(*)     OVER ()                                                          AS total_vendedores,
  SUM(receita) OVER (ORDER BY receita DESC
                     ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)          AS receita_acumulada,
  SUM(receita) OVER ()                                                          AS receita_total
FROM (
  SELECT seller_id, SUM(preco) AS receita
  FROM mvp.gold.fato_itens_pedido
  GROUP BY seller_id
) r;

SELECT
  MAX(total_vendedores)                                                          AS total_vendedores,
  MIN(CASE WHEN receita_acumulada / receita_total >= 0.5 THEN posicao END)       AS vendedores_para_50pct,
  ROUND(MIN(CASE WHEN receita_acumulada / receita_total >= 0.5 THEN posicao END)
        / MAX(total_vendedores) * 100, 1)                                        AS pct_vendedores_50pct,
  MIN(CASE WHEN receita_acumulada / receita_total >= 0.8 THEN posicao END)       AS vendedores_para_80pct,
  ROUND(MIN(CASE WHEN receita_acumulada / receita_total >= 0.8 THEN posicao END)
        / MAX(total_vendedores) * 100, 1)                                        AS pct_vendedores_80pct
FROM v_ranking_vendedores;

In [0]:
%sql
-- Pontos da curva de Pareto para o gráfico
SELECT
  ROUND(posicao / total_vendedores * 100, 2)          AS pct_vendedores,
  ROUND(receita_acumulada / receita_total * 100, 2)   AS pct_receita_acumulada
FROM v_ranking_vendedores
ORDER BY posicao;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
  ROUND(MAX(CASE WHEN posicao / total_vendedores <= 0.4 THEN receita_acumulada / receita_total END) * 100, 1)       AS pct_receita_top40,
  ROUND((1 - MAX(CASE WHEN posicao / total_vendedores <= 0.5 THEN receita_acumulada / receita_total END)) * 100, 1) AS pct_receita_metade_inferior
FROM v_ranking_vendedores;

**Resultado e interpretação — P7**

- Dos **3095** vendedores ativos, apenas **130 (4.2 %)** respondem por metade da receita, e **544 (17.6 %)** por 80%.
- A concentração é, portanto, **maior do que a sugerida pela regra 80/20**: menos de um quinto dos vendedores já gera 80% do faturamento.
- A curva sobe de forma quase vertical no início: os primeiros 4.2 % dos vendedores levam a receita acumulada a 50%. Depois ela se achata rapidamente. Os 40% maiores vendedores já acumulam **94.4 %** da receita, e a metade inferior da base, cerca de 1.500 vendedores, responde por apenas **3.2 %**.

---
## Discussão geral

O problema proposto no MVP00 era identificar quais fatores logísticos, geográficos e comerciais explicam a
satisfação do cliente e a concentração de receita. As sete respostas convergem em três conclusões.

### 1. A logística é o principal vetor de satisfação

O cumprimento do prazo é o fator com maior efeito observado sobre a avaliação: a nota média cai de **4.29** para
**2.27** quando o pedido atrasa, e a proporção de notas baixas sobe de **9.3%** para **62.4%** (P1). O dano não é
gradual: a partir de **4 dias de atraso**, a nota média fica abaixo de 3 e as notas baixas passam a ser maioria, o que
define um limite operacional claro.

Esse risco é geograficamente desigual. Clientes de RR esperam em média **29.3 dias**, contra **8.7 dias** em SP, e
nas UFs de maior prazo o frete pesa **28.3%** do valor do pedido, contra **19.5%** no Sudeste (P2): quem espera mais
também paga proporcionalmente mais. Parte dessa diferença se explica pela origem do envio: pedidos interestaduais
levam, em mediana, **13 dias**, quase o dobro dos **7 dias** dos intraestaduais, diferença estatisticamente
significativa (p < 0,001) e também relevante na prática (P3).

### 2. A receita é concentrada, sobretudo em vendedores

Oito categorias concentram **54.4%** da receita, lideradas por beleza_saude (**9.26%**) (P4). A concentração entre
vendedores é ainda mais acentuada: **4.2%** dos vendedores geram metade da receita e **17.6%** geram 80%, acima do
padrão 80/20 (P7).

Entre as categorias de alto volume, sete ficam abaixo da média geral de nota. Chama atenção a presença de
`sem_categoria` nesse grupo: produtos com cadastro incompleto também aparecem associados a experiências piores,
um ponto em que a qualidade dos dados e a qualidade do serviço se encontram.

### 3. O crescimento veio do volume, e isso pressiona a logística

Entre janeiro e agosto, a receita de 2018 superou a de 2017 em **140.36%**, enquanto o ticket médio variou apenas
**1.31%**: a plataforma cresceu vendendo mais pedidos, não pedidos maiores (P5). O cartão de crédito domina, com
**75.5%** dos pedidos, e o ticket mediano sobe de **R$ 71.63** à vista para **R$ 213.74** em 11 parcelas ou mais,
com correlação moderada (r = 0.369) (P6).

As conclusões se conectam: como o crescimento veio do volume, cada ponto percentual de atraso afeta um número
cada vez maior de clientes. Escalar a operação sem resolver os gargalos logísticos do primeiro ponto tende a
ampliar, e não diluir, o problema de satisfação.

### Considerações

- As relações observadas são **associações, não causalidade**: atraso e nota baixa podem ter causas em comum, e é
  provável que compras caras levem ao parcelamento, e não o contrário.
- UFs com poucos pedidos, como RR, têm médias menos estáveis.
- Com dezenas de milhares de pedidos, testes estatísticos detectam diferenças mínimas; por isso a relevância foi
  avaliada também pelo tamanho do efeito (diferença de medianas).
- O indicador interestadual é uma aproximação: a distância real entre vendedor e cliente não foi considerada,
  pois a geolocalização ficou fora do modelo.
- A base cobre 2016–2018, é estática, e a série temporal exclui os meses de borda com volume residual.